# AeroPulse Model 1 — Hyper-local PM2.5 Estimator

**Purpose:** train a 1-km PM2.5 estimator using ground observations + meteorology + fire evidence, then predict a new grid/time.

This follows the LLD: LightGBM/XGBoost first, with ground AQ, AOD/satellite signals where available, weather, fire activity, land/urban context and historical PM2.5. AOD is treated as a feature, never as surface PM2.5 itself.

**Important:** the notebook uses OpenAQ for reproducible public ground observations and Open-Meteo/ERA5-accessible weather for a runnable baseline. FIRMS is integrated when `FIRMS_MAP_KEY` is configured. The production adapter should replace the weather connector with IMD while preserving the canonical feature contract.

In [ ]:
!pip -q install pandas numpy scikit-learn lightgbm joblib requests pyarrow

In [ ]:
# Core
import io
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path(os.getenv("AEROPULSE_DATA_DIR", "./data"))
ARTIFACT_DIR = Path(os.getenv("AEROPULSE_ARTIFACT_DIR", "./artifacts"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REGION = {
    "min_lat": float(os.getenv("AEROPULSE_MIN_LAT", "28.0")),
    "max_lat": float(os.getenv("AEROPULSE_MAX_LAT", "32.8")),
    "min_lon": float(os.getenv("AEROPULSE_MIN_LON", "74.0")),
    "max_lon": float(os.getenv("AEROPULSE_MAX_LON", "78.5")),
}

OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY", "")
FIRMS_MAP_KEY = os.getenv("FIRMS_MAP_KEY", "")

def http_get_json(url, params=None, headers=None, timeout=60):
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    return r.json()

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def require_env(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

print("AeroPulse ML environment initialized")
print("Region:", REGION)

In [ ]:
# OpenAQ v3 connector. OpenAQ exposes original measurements plus hourly aggregates.
# Use API key through OPENAQ_API_KEY.
OPENAQ_BASE = "https://api.openaq.org/v3"

def openaq_locations(bbox=REGION, limit=1000):
    params = {
        "bbox": f'{bbox["min_lon"]},{bbox["min_lat"]},{bbox["max_lon"]},{bbox["max_lat"]}',
        "limit": min(limit, 1000),
        "page": 1,
    }
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    return http_get_json(f"{OPENAQ_BASE}/locations", params=params, headers=headers)

def openaq_location_hours(location_id, date_from, date_to, limit=1000):
    headers = {"X-API-Key": require_env("OPENAQ_API_KEY")}
    params = {
        "datetime_from": pd.Timestamp(date_from, tz="UTC").isoformat(),
        "datetime_to": pd.Timestamp(date_to, tz="UTC").isoformat(),
        "limit": limit,
        "page": 1,
    }
    return http_get_json(f"{OPENAQ_BASE}/locations/{int(location_id)}/hours",
                         params=params, headers=headers)

def discover_pm25_locations(max_locations=100):
    raw = openaq_locations(limit=max_locations)
    rows = raw.get("results", [])
    out = []
    for x in rows:
        params = x.get("parameters", [])
        names = {str(p.get("name", "")).lower() for p in params}
        if "pm25" in names or "pm2.5" in names:
            coords = x.get("coordinates") or {}
            if coords.get("latitude") is not None and coords.get("longitude") is not None:
                out.append({
                    "location_id": x["id"],
                    "name": x.get("name"),
                    "lat": coords["latitude"],
                    "lon": coords["longitude"],
                    "parameters": list(names),
                })
    return pd.DataFrame(out)

def normalize_openaq_hours(payload):
    rows = payload.get("results", [])
    out = []
    for r in rows:
        period = r.get("period", {})
        dt = (period.get("datetimeFrom") or {}).get("utc")
        param = r.get("parameter", {})
        out.append({
            "timestamp": pd.to_datetime(dt, utc=True, errors="coerce"),
            "parameter": str(param.get("name", "")).lower(),
            "value": safe_float(r.get("value")),
            "unit": r.get("unit"),
            "location_id": r.get("locationsId"),
        })
    return pd.DataFrame(out).dropna(subset=["timestamp"])

In [ ]:
# Open-Meteo is used here as a reproducible public weather/ERA5 access path.
# In production, replace this adapter with the IMD connector defined in the AeroPulse LLD.
OPEN_METEO_ARCHIVE = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_FORECAST = "https://api.open-meteo.com/v1/forecast"

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "surface_pressure",
    "precipitation", "wind_speed_10m", "wind_direction_10m",
    "boundary_layer_height"
]

def weather_archive(lat, lon, start_date, end_date):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": pd.Timestamp(start_date).date().isoformat(),
        "end_date": pd.Timestamp(end_date).date().isoformat(),
        "hourly": ",".join(WEATHER_VARS),
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_ARCHIVE, params=params)

def weather_forecast(lat, lon, forecast_days=2):
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": ",".join(WEATHER_VARS),
        "forecast_days": forecast_days,
        "timezone": "UTC",
    }
    return http_get_json(OPEN_METEO_FORECAST, params=params)

def normalize_weather(payload):
    h = payload.get("hourly", {})
    df = pd.DataFrame(h)
    if "time" in df:
        df["timestamp"] = pd.to_datetime(df.pop("time"), utc=True)
    return df

In [ ]:
FIRMS_BASE = "https://firms.modaps.eosdis.nasa.gov/api"

def firms_area_csv(source, west, south, east, north, days=1):
    key = require_env("FIRMS_MAP_KEY")
    area = f"{west},{south},{east},{north}"
    url = f"{FIRMS_BASE}/area/csv/{key}/{source}/{area}/{int(days)}"
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def normalize_firms(df):
    if df.empty:
        return df
    out = df.copy()
    out["timestamp"] = pd.to_datetime(
        out["acq_date"].astype(str) + " " + out["acq_time"].astype(str).str.zfill(4),
        format="%Y-%m-%d %H%M", utc=True, errors="coerce"
    )
    out["frp"] = pd.to_numeric(out.get("frp"), errors="coerce")
    out["confidence_num"] = pd.to_numeric(out.get("confidence"), errors="coerce")
    return out.dropna(subset=["timestamp", "latitude", "longitude"])

In [ ]:
def grid_id(lat, lon, resolution_km=1.0):
    # Approximate 1-km grid using latitude-dependent longitude scaling.
    lat_step = resolution_km / 111.32
    lon_step = resolution_km / (111.32 * np.cos(np.deg2rad(np.clip(lat, -89, 89))))
    ilat = np.floor(lat / lat_step).astype(int)
    ilon = np.floor(lon / lon_step).astype(int)
    return pd.Series(ilat).astype(str) + "_" + pd.Series(ilon).astype(str)

def add_grid(df, lat_col="lat", lon_col="lon"):
    out = df.copy()
    out["grid_id"] = grid_id(out[lat_col].astype(float), out[lon_col].astype(float))
    return out

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dphi = np.deg2rad(lat2-lat1)
    dl = np.deg2rad(lon2-lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

## 1. Acquire historical training data

The training target is the ground PM2.5 observation. We discover PM2.5-capable locations in the pilot bounding box and collect hourly history.

For production, CPCB CAAQMS should be the preferred ground-truth source, with OpenAQ as a secondary interoperability source.

In [ ]:
locations = discover_pm25_locations(max_locations=100)
print("PM2.5 locations discovered:", len(locations))
display(locations.head(20))
if locations.empty:
    raise RuntimeError("No PM2.5 locations found. Check OPENAQ_API_KEY and region settings.")

In [ ]:
# Configurable training window. Start smaller for a first run.
END = pd.Timestamp.utcnow().floor("h")
START = END - pd.Timedelta(days=int(os.getenv("TRAIN_DAYS", "30")))

records = []
for _, loc in locations.head(int(os.getenv("MAX_STATIONS", "20"))).iterrows():
    try:
        payload = openaq_location_hours(int(loc.location_id), START, END)
        d = normalize_openaq_hours(payload)
        d = d[d["parameter"].isin(["pm25", "pm2.5"])]
        if not d.empty:
            d["lat"], d["lon"] = loc.lat, loc.lon
            d["station_name"] = loc["name"]
            records.append(d)
    except Exception as e:
        print("station failed", loc.location_id, e)

aq = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
print("AQ rows:", len(aq))
display(aq.head())
if aq.empty:
    raise RuntimeError("No PM2.5 history returned.")

## 2. Feature engineering

Features are explicitly point-in-time safe:

- lagged PM2.5 from the same station
- rolling PM2.5 statistics
- cyclic hour/day-of-year
- wind converted to U/V
- temperature, humidity, pressure, rainfall and boundary-layer height
- optional FIRMS fire count/FRP in a configurable radius
- spatial station/grid identity

Satellite AOD/trace gases and static land-use/elevation fields are accepted by the feature schema; if their connector is not configured, they remain missing and are handled by the model pipeline. This prevents silently substituting a proxy for AOD.

In [ ]:
def make_weather_for_station(lat, lon, start, end):
    w = normalize_weather(weather_archive(lat, lon, start, end))
    if w.empty:
        return w
    w["lat"], w["lon"] = lat, lon
    return w

weather_parts = []
for (lat, lon), _ in aq.groupby(["lat", "lon"]):
    try:
        weather_parts.append(make_weather_for_station(lat, lon, START.date(), END.date()))
    except Exception as e:
        print("weather failed", lat, lon, e)

weather = pd.concat(weather_parts, ignore_index=True) if weather_parts else pd.DataFrame()
print("Weather rows:", len(weather))

In [ ]:
aq = aq.sort_values(["location_id", "timestamp"]).copy()
aq["pm25"] = pd.to_numeric(aq["value"], errors="coerce")
aq = aq.dropna(subset=["pm25"])
aq["hour"] = aq.timestamp.dt.hour
aq["doy"] = aq.timestamp.dt.dayofyear
aq["sin_hour"] = np.sin(2*np.pi*aq.hour/24)
aq["cos_hour"] = np.cos(2*np.pi*aq.hour/24)
aq["sin_doy"] = np.sin(2*np.pi*aq.doy/365.25)
aq["cos_doy"] = np.cos(2*np.pi*aq.doy/365.25)

g = aq.groupby("location_id")["pm25"]
aq["pm25_lag1"] = g.shift(1)
aq["pm25_lag3"] = g.shift(3)
aq["pm25_roll6"] = g.transform(lambda s: s.shift(1).rolling(6, min_periods=3).mean())
aq["pm25_roll24"] = g.transform(lambda s: s.shift(1).rolling(24, min_periods=6).mean())

weather["wind_direction_10m"] = pd.to_numeric(weather["wind_direction_10m"], errors="coerce")
wd = np.deg2rad(weather["wind_direction_10m"])
weather["wind_u"] = -np.sin(wd) * weather["wind_speed_10m"]
weather["wind_v"] = -np.cos(wd) * weather["wind_speed_10m"]

features = aq.merge(weather, on=["timestamp", "lat", "lon"], how="left")
features["grid_id"] = grid_id(features["lat"], features["lon"])

# Fire evidence: only enabled when a FIRMS key is present.
features["fire_count_20km"] = 0.0
features["fire_frp_20km"] = 0.0
if FIRMS_MAP_KEY:
    try:
        fire = normalize_firms(
            firms_area_csv(
                "VIIRS_NOAA20_NRT",
                REGION["min_lon"], REGION["min_lat"],
                REGION["max_lon"], REGION["max_lat"], days=7
            )
        )
        if not fire.empty:
            # Spatial join without requiring geopandas: approximate radius.
            vals = []
            for _, r in features[["timestamp","lat","lon"]].drop_duplicates().iterrows():
                recent = fire[
                    (fire["timestamp"] <= r.timestamp) &
                    (fire["timestamp"] > r.timestamp - pd.Timedelta(hours=24))
                ]
                if recent.empty:
                    vals.append((r.timestamp, r.lat, r.lon, 0, 0))
                else:
                    d = haversine_km(r.lat, r.lon, recent.latitude, recent.longitude)
                    hit = recent[d <= 20]
                    vals.append((r.timestamp, r.lat, r.lon, len(hit), hit.frp.sum()))
            fire_features = pd.DataFrame(vals, columns=["timestamp","lat","lon","fire_count_20km","fire_frp_20km"])
            features = features.drop(columns=["fire_count_20km","fire_frp_20km"]).merge(
                fire_features, on=["timestamp","lat","lon"], how="left"
            )
    except Exception as e:
        print("FIRMS optional feature disabled:", e)

features["fire_count_20km"] = features["fire_count_20km"].fillna(0)
features["fire_frp_20km"] = features["fire_frp_20km"].fillna(0)

print(features.shape)
display(features.head())

## 3. Leakage-safe split and model

Do **not** use a random split. The LLD requires spatial and temporal holdouts because adjacent locations/times are correlated.

This notebook uses the final 20% of time as a temporal test set and a held-out subset of stations as an additional spatial stress test when enough stations exist.

In [ ]:
FEATURES = [
    "pm25_lag1","pm25_lag3","pm25_roll6","pm25_roll24",
    "temperature_2m","relative_humidity_2m","surface_pressure",
    "precipitation","wind_speed_10m","wind_direction_10m",
    "wind_u","wind_v","boundary_layer_height",
    "sin_hour","cos_hour","sin_doy","cos_doy",
    "fire_count_20km","fire_frp_20km",
    "lat","lon"
]
TARGET = "pm25"

model_df = features.dropna(subset=[TARGET]).copy()
cut = model_df["timestamp"].quantile(0.80)
train = model_df[model_df.timestamp < cut].copy()
test = model_df[model_df.timestamp >= cut].copy()

# Keep only columns that actually exist.
FEATURES = [c for c in FEATURES if c in model_df.columns]
print("train/test:", train.shape, test.shape)
print("features:", FEATURES)

In [ ]:
import joblib
from lightgbm import LGBMRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

imputer = SimpleImputer(strategy="median")
Xtr = imputer.fit_transform(X_train)
Xte = imputer.transform(X_test)

model = LGBMRegressor(
    n_estimators=700, learning_rate=0.03, num_leaves=63,
    subsample=0.85, colsample_bytree=0.85, random_state=SEED,
    objective="regression"
)
model.fit(Xtr, y_train)

pred = model.predict(Xte)
metrics = {
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": mean_squared_error(y_test, pred, squared=False),
    "R2": r2_score(y_test, pred),
}
metrics

## 4. Validation diagnostics

The notebook reports MAE/RMSE/R² and checks error by pollution regime. For a production release, add explicit geography holdouts, season holdouts, calibration intervals and CPCB-based external validation.

In [ ]:
test_eval = test[["timestamp","lat","lon","pm25"]].copy()
test_eval["prediction"] = pred
test_eval["abs_error"] = np.abs(test_eval.pm25 - test_eval.prediction)
test_eval["pm25_bin"] = pd.cut(
    test_eval.pm25, bins=[-np.inf, 30, 60, 90, 150, np.inf],
    labels=["low","moderate","high","very_high","extreme"]
)
print(pd.DataFrame([metrics]))
display(test_eval.groupby("pm25_bin")["abs_error"].agg(["count","mean","median"]))

In [ ]:
# Persist model + feature contract.
bundle = {
    "model": model,
    "imputer": imputer,
    "features": FEATURES,
    "training_cutoff": str(cut),
    "region": REGION,
    "model_name": "aeropulse_pm25_estimator",
    "version": "1.0.0",
    "metrics": metrics,
}
joblib.dump(bundle, ARTIFACT_DIR / "pm25_estimator.joblib")
with open(ARTIFACT_DIR / "pm25_estimator_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved:", ARTIFACT_DIR / "pm25_estimator.joblib")

## 5. Live prediction

This section demonstrates the online path: obtain the newest OpenAQ observation, current/forecast weather and recent FIRMS evidence, build exactly the same feature contract, and predict PM2.5.

For production, the live inference service should consume AeroPulse's normalized 1-km feature record rather than calling external providers synchronously.

In [ ]:
bundle = joblib.load(ARTIFACT_DIR / "pm25_estimator.joblib")

latest_locations = locations.head(min(10, len(locations)))
live_rows = []
for _, loc in latest_locations.iterrows():
    try:
        payload = http_get_json(
            f"{OPENAQ_BASE}/locations/{int(loc.location_id)}/latest",
            headers={"X-API-Key": require_env("OPENAQ_API_KEY")}
        )
        for r in payload.get("results", []):
            p = str((r.get("parameter") or {}).get("name","")).lower()
            if p in ("pm25","pm2.5"):
                dt = (r.get("datetime") or r.get("period",{}).get("datetimeTo") or {})
                dt = dt.get("utc") if isinstance(dt, dict) else dt
                live_rows.append({
                    "timestamp": pd.to_datetime(dt, utc=True, errors="coerce"),
                    "location_id": loc.location_id, "lat": loc.lat, "lon": loc.lon,
                    "pm25": safe_float(r.get("value"))
                })
    except Exception as e:
        print("latest failed", loc.location_id, e)

live = pd.DataFrame(live_rows).dropna(subset=["timestamp"])
print("Live observations:", len(live))
display(live.head())

In [ ]:
# Build a minimal live feature row for the newest station observation.
# Historical lag/rolling values should come from TimescaleDB in production.
if not live.empty:
    r = live.sort_values("timestamp").iloc[-1]
    w = normalize_weather(weather_forecast(r.lat, r.lon, forecast_days=2))
    w0 = w.iloc[(w.timestamp - r.timestamp).abs().argmin()].to_dict()

    x = {c: np.nan for c in bundle["features"]}
    x.update({
        "pm25_lag1": r.pm25,
        "pm25_lag3": r.pm25,
        "pm25_roll6": r.pm25,
        "pm25_roll24": r.pm25,
        "temperature_2m": w0.get("temperature_2m"),
        "relative_humidity_2m": w0.get("relative_humidity_2m"),
        "surface_pressure": w0.get("surface_pressure"),
        "precipitation": w0.get("precipitation"),
        "wind_speed_10m": w0.get("wind_speed_10m"),
        "wind_direction_10m": w0.get("wind_direction_10m"),
        "boundary_layer_height": w0.get("boundary_layer_height"),
        "wind_u": -np.sin(np.deg2rad(w0.get("wind_direction_10m", np.nan))) * w0.get("wind_speed_10m", np.nan),
        "wind_v": -np.cos(np.deg2rad(w0.get("wind_direction_10m", np.nan))) * w0.get("wind_speed_10m", np.nan),
        "sin_hour": np.sin(2*np.pi*r.timestamp.hour/24),
        "cos_hour": np.cos(2*np.pi*r.timestamp.hour/24),
        "sin_doy": np.sin(2*np.pi*r.timestamp.dayofyear/365.25),
        "cos_doy": np.cos(2*np.pi*r.timestamp.dayofyear/365.25),
        "lat": r.lat, "lon": r.lon
    })
    live_X = pd.DataFrame([x])[bundle["features"]]
    live_pred = bundle["model"].predict(bundle["imputer"].transform(live_X))[0]
    print({"timestamp": str(r.timestamp), "lat": r.lat, "lon": r.lon, "pm25_estimate": float(live_pred)})
else:
    print("No live observation available.")